# Étape 2 — Prétraitement et équilibrage (version Jupyter)

Ce notebook est l'équivalent de `02_pretraitement.py`. Il part de l'échantillon
de l'étape 1, nettoie les colonnes, sépare train et test, puis compare trois
façons de traiter le déséquilibre des classes.

Règles conservées :

- graine `RANDOM_STATE = 42` partout ;
- déduplication avant le split pour éviter qu'une même ligne soit dans les deux
  jeux ;
- imputation et rééchantillonnage uniquement dans les folds de validation
  croisée ;
- jeu de test intact jusqu'à l'étape 5.

In [ ]:
import os
import sys
from pathlib import Path

# Jupyter ouvre le notebook depuis notebooks/ : on remonte jusqu'à config.py
dossier = Path.cwd()
while dossier != dossier.parent and not (dossier / "config.py").exists():
    dossier = dossier.parent
os.chdir(dossier)
if str(dossier) not in sys.path:
    sys.path.insert(0, str(dossier))
print("Dossier du projet :", dossier)

In [ ]:
%matplotlib inline

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler

from config import (RANDOM_STATE, RISKY_COLUMNS, RARE_FAMILIES,
                    PROCESSED_DIR, FIGURES_DIR, TABLES_DIR, SAMPLE_FILE)

print("Graine utilisee :", RANDOM_STATE)

## 1. Chargement de l'échantillon

On relit le fichier écrit à l'étape 1. Aucun CSV brut n'est touché ici.

In [ ]:
df = pd.read_csv(PROCESSED_DIR / SAMPLE_FILE, low_memory=False)
n_depart = len(df)
print("Départ :", n_depart, "lignes et", df.shape[1], "colonnes")

## 2. Colonnes identifiantes, constantes et valeurs infinies

Les identifiants (`Flow ID`, IP, ports, `Timestamp`) sont retirés pour ne pas
mémoriser une machine ou un horodatage. Les colonnes constantes relevées à
l'étape 1 partent aussi. Les débits infinis deviennent des `NaN` : l'imputation
sera faite plus tard, sur le train seulement.

In [ ]:
colonnes_a_retirer = [c for c in RISKY_COLUMNS if c in df.columns]
df = df.drop(columns=colonnes_a_retirer)
print("Colonnes identifiantes retirées :", colonnes_a_retirer)

uniques = df.nunique(dropna=False)
constantes = list(uniques[uniques <= 1].index)
df = df.drop(columns=constantes)
print("Colonnes constantes retirées :", len(constantes))

colonnes_num = list(df.select_dtypes(include="number").columns)
nb_infinis = int(np.isinf(df[colonnes_num].to_numpy(dtype=float)).sum())
df[colonnes_num] = df[colonnes_num].replace([np.inf, -np.inf], np.nan)
print("Infinis remplacés par NaN :", nb_infinis)

## 3. Doublons et familles trop rares

Les doublons sont supprimés **avant** le split, sinon une ligne identique
pourrait se retrouver à la fois en train et en test. Les familles
`Infiltration` (36) et `Heartbleed` (11) sont mises de côté dans
`attaques_rares.csv` : trop peu d'exemples pour les évaluer.

In [ ]:
avant_dedup = len(df)
df = df.drop_duplicates(subset=colonnes_num + ["Label"]).reset_index(drop=True)
apres_dedup = len(df)
print("Doublons supprimés :", avant_dedup - apres_dedup)

rares = df[df["Family"].isin(RARE_FAMILIES)].copy()
rares.to_csv(PROCESSED_DIR / "attaques_rares.csv", index=False)
df = df[~df["Family"].isin(RARE_FAMILIES)].reset_index(drop=True)
print("Familles rares mises de côté :", len(rares), "(attaques_rares.csv)")

## 4. Séparation train / test

Split stratifié 80/20 avec la graine 42. On écrit `train.csv` et `test.csv`.
À partir d'ici, le test n'est plus utilisé avant l'évaluation finale.

In [ ]:
y = df["Family"]
X = df.drop(columns=["Family", "Label", "source_file", "jour"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Train :", len(X_train), "lignes / Test :", len(X_test), "lignes")
print("Classes du train :", y_train.value_counts().to_dict())

resume = pd.DataFrame({
    "etape": ["échantillon", "après colonnes à risque", "après colonnes constantes",
              "après doublons", "après familles rares", "train", "test"],
    "lignes": [n_depart, n_depart, n_depart, apres_dedup, len(df), len(X_train), len(X_test)],
})
resume.to_csv(TABLES_DIR / "resume_pretraitement.csv", index=False)

train = X_train.copy()
train["Family"] = y_train
test = X_test.copy()
test["Family"] = y_test
train.to_csv(PROCESSED_DIR / "train.csv", index=False)
test.to_csv(PROCESSED_DIR / "test.csv", index=False)
resume

## 5. Comparaison de trois stratégies d'équilibrage

On teste `class_weight`, le sous-échantillonnage et SMOTE avec une régression
logistique, en validation croisée 5 folds sur le train uniquement.

Le sampleur est dans le pipeline `imblearn` : à chaque fold, il ne voit que les
données d'entraînement du fold, jamais la partie réservée au test du fold, et
jamais notre jeu de test final. En multi-classes, `SMOTE` attend un dictionnaire
`{classe : effectif voulu}`.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

comptage = y_train.value_counts()
majoritaire = comptage.idxmax()
cible_smote = {}
for classe, nb in comptage.drop(majoritaire).items():
    cible_smote[classe] = min(int(0.2 * comptage[majoritaire]), 10 * int(nb))

strategies = {
    "class_weight": ("passthrough", {"class_weight": "balanced"}),
    "sous-échantillonnage": (
        RandomUnderSampler(sampling_strategy={majoritaire: 20000}, random_state=RANDOM_STATE), {}),
    "SMOTE": (SMOTE(sampling_strategy=cible_smote, random_state=RANDOM_STATE), {}),
}
resultats = []

for nom, (sampleur, parametres) in strategies.items():
    pipe = Pipeline([
        ("imputation", SimpleImputer(strategy="median")),
        ("scaling", StandardScaler()),
        ("equilibrage", sampleur),
        ("modele", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                                      **parametres)),
    ])
    debut = time.time()
    scores = cross_validate(pipe, X_train, y_train, cv=cv,
                            scoring=["f1_macro", "accuracy"])
    resultats.append({
        "strategie": nom,
        "f1_macro_moy": round(scores["test_f1_macro"].mean(), 4),
        "f1_macro_std": round(scores["test_f1_macro"].std(), 4),
        "accuracy_moy": round(scores["test_accuracy"].mean(), 4),
        "temps_s": round(time.time() - debut, 1),
    })
    print(nom, "F1 macro =", resultats[-1]["f1_macro_moy"],
          "+/-", resultats[-1]["f1_macro_std"])

In [ ]:
comparaison = pd.DataFrame(resultats)
comparaison.to_csv(TABLES_DIR / "comparaison_equilibrage.csv", index=False)
comparaison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(comparaison["strategie"], comparaison["f1_macro_moy"],
       yerr=comparaison["f1_macro_std"], capsize=4, color="steelblue")
ax.set_ylim(0, 1)
ax.set_title("F1 macro en validation croisée (train uniquement)")
ax.set_ylabel("F1 macro")
ax.tick_params(axis="x", rotation=15)
plt.show()
fig.savefig(FIGURES_DIR / "comparaison_equilibrage.png", dpi=120, bbox_inches="tight")
plt.close(fig)

## Résumé de l'étape 2 (version Jupyter)

Résultats identiques à ceux du script `02_pretraitement.py` :

- 141 581 lignes au départ ; 5 549 doublons supprimés ;
- 47 lignes rares mises de côté ; train 108 788 / test 27 197 ;
- F1 macro en CV 5 folds : `class_weight` 0,5616 ; sous-échantillonnage 0,6395 ;
  SMOTE 0,6544 ;
- l'accuracy de 0,84 du premier cas cache un F1 macro de 0,56 à cause de la
  majorité `BENIGN` ;
- le jeu de test n'a pas été utilisé dans cette étape.

In [ ]:
print("\nLe jeu de test n'a pas été utilisé ici. Fin de l'étape 2 (version Jupyter).")